# 09 - IO-Aware Attention

FlashAttention made exact attention faster by refusing to write a matrix to
memory. Understanding why is the cleanest introduction to a rule that governs
modern inference: arithmetic is cheap, moving bytes is not.

## The apparent contradiction

![IO-aware attention tiling](https://bankoti.github.io/llm-quest/content/images/c2/io_aware.svg)


The attention equation creates a score matrix with `T * T` entries per head. A
straight implementation writes that matrix to high-bandwidth memory, reads it for
softmax, writes probabilities, then reads them again to multiply by values. For
long sequences, moving these intermediates can cost more time than the arithmetic.

FlashAttention changes this memory schedule while computing exact attention. It
does **not** replace softmax attention with an approximation.

## Online softmax

For scores `s`, ordinary softmax uses:

```text
m = max(s)
l = sum(exp(s - m))
softmax(s_i) = exp(s_i - m) / l
```

Suppose scores arrive in blocks. Maintain a running maximum `m_old`, running
normalizer `l_old`, and partial weighted output `o_old`. For a new block with
maximum `m_block`, choose:

```text
m_new = max(m_old, m_block)
l_new = exp(m_old - m_new) * l_old
      + sum(exp(s_block - m_new))
```

The old contribution is rescaled because the numerical-stability reference
maximum changed. The output accumulator is rescaled the same way. This makes it
possible to tile queries, keys, and values through fast on-chip memory without
materializing the full score or probability matrices in external memory.

## What remains identical

- causal or local masking semantics;
- scaling by `sqrt(head_size)`;
- row-wise softmax;
- the final weighted sum;
- gradients, up to normal floating-point tolerance.

An optimized kernel is correct only if it preserves those semantics. Benchmark
speed and numerical agreement separately.

## Why hardware details matter

Arithmetic throughput, memory bandwidth, on-chip SRAM, tensor-core formats, warp
scheduling, and supported head dimensions determine realized speed. A kernel that
wins on one GPU or dtype may not win on a CPU, an older GPU, short sequences, or
unusual masking. FlashAttention-3 adds Hopper-specific scheduling and low-precision
techniques; that is a systems advance, not a new language-model objective.

## Worked traffic estimate

For one FP16 attention head with `T=4096`, storing only a dense score matrix costs:

```text
4096 * 4096 * 2 bytes = 32 MiB
```

Multiply by batch and heads, then add probabilities and gradients. This simple
estimate explains why avoiding external-memory intermediates matters even though
the number of dot products remains quadratic.

## Experiment

Complete the memory accounting, then implement `tiled_attention(q,k,v,block_size)`.
This CPU reference validates the online-softmax algorithm; it is not a fused GPU
kernel. Storage estimates are not measured traffic or wall-clock speed.

If you have supported hardware, compare PyTorch's explicit attention against
`scaled_dot_product_attention`. Verify output closeness before timing warm runs.
Synchronize the accelerator around measurements and report shapes, dtype, device,
kernel backend, median, and tail latency.

## Failure modes

- Calling every fused attention implementation "FlashAttention."
- Reporting one unsynchronized GPU timing.
- Treating lower allocated memory as proof of equal output.
- Assuming an IO-aware exact kernel makes quadratic compute linear.
- Ignoring padding, mask shape, dropout, dtype, or backward-pass differences.

## Exit check

Explain how exact attention can avoid storing the full score matrix, and name one
architecture property and one hardware property that determine the speedup.

Primary source: [FlashAttention-3](https://arxiv.org/abs/2407.08608).

## Implement the recurrence

For each query tile, initialize one maximum `m=-inf` and exponential sum `l=0`
per row, and weighted value sum `a=0` with shape `(query_rows, value_width)`.
Visit key tiles up to the end of this query tile. Use GLOBAL position indices
to set future scores to `-inf`. The diagonal remains visible.

```text
S = Q_tile @ K_tile.T / sqrt(D), with causal mask
m_new = maximum(m, row_max(S))
alpha = exp(m - m_new)
P = exp(S - m_new[:, None])
l_new = alpha * l + row_sum(P)
a_new = alpha[:, None] * a + P @ V_tile
```

Assign the new statistics after each tile. Return `a/l[:,None]` only after all
allowed tiles. The first key tile includes at least one visible key for each
query row, so its maximum is finite. Skipping fully future tiles also avoids
the undefined expression `-inf - -inf` in an empty accumulator.

For scores `[1,2]` and values `[10,20]`, `m=2`, `l=1+exp(-1)`, and
`a=20+10*exp(-1)`. A later tile with maximum 4 changes the scale: multiply
both previous sums by `exp(2-4)` before adding new contributions. Equal-weight
averaging of two tile outputs would discard their different probability mass.

Implement query tiling as well as key tiling. With tile size B, score storage is
at most B squared (when T>B), with additional row accumulators and input/output
arrays. The grader checks multiple tile sizes, non-divisible lengths, large
logits, and isolation from future values. It cannot prove memory scheduling from
outputs alone; inspect allocations and compare peak memory in the notebook.

**Challenge:** Deliberately remove the `alpha` correction. Explain why a rising
maximum breaks the result even when no NaN appears. The worked solution is
available below the coding arena.

Primary algorithm: [Dao et al., FlashAttention, Algorithm 1](https://arxiv.org/abs/2205.14135).


## Work first, then compare

CPU runtime is sufficient for the exercises. NumPy and PyTorch are available in Colab. Run the starter and grading cells after implementing the TODOs. An untouched starter is expected to fail. The reference solution below replaces the functions only when you deliberately run that cell. Then rerun the checks. No account, key, model download, or paid GPU is required.


In [ ]:
"""IO-aware attention: memory accounting, then exact tiled causal attention.

Implement the recurrence in the lesson. This NumPy reference teaches the
algorithm; it is not a fused GPU kernel and has no speedup guarantee.
"""
import numpy as np
def score_matrix_bytes(batch:int, heads:int, tokens:int, bytes_per_value:int)->int:
    """Naive attention materialises (tokens,tokens) per head. Total bytes."""
    raise NotImplementedError

def tiled_attention(q, k, v, block_size=2):
    """Return causal softmax(q @ k.T / sqrt(D)) @ v without a full T x T matrix.

    q,k: finite arrays (T,D); v: (T,Dv); T,D,Dv > 0.
    block_size: positive integer. Process query and key blocks; handle tails.
    Track each row's maximum m, exponential sum l, and weighted value sum a.
    Skip key blocks entirely in the future; mask future entries within a tile.
    Use float64 accumulators and return (T,Dv). Raise ValueError for block_size<=0.
    """
    raise NotImplementedError

def tile_bytes(query_rows:int, key_rows:int, head_size:int, bytes_per_value:int)->int:
    """One tile: Q tile + K tile + V tile + score tile."""
    raise NotImplementedError


In [ ]:
naive=score_matrix_bytes(1,1,4096,2)
assert naive==32*1024*1024,f"4096^2*2=32MiB, got {naive}"
tiled=tile_bytes(64,64,128,2)
assert tiled<naive,"one tile < full matrix"
assert tiled==64*128*2+64*128*2+64*128*2+64*64*2,f"tile bytes: {tiled}"
assert score_matrix_bytes(1,1,8192,2)==4*naive,"doubling tokens->4x memory"
print("✓ naive score matrix correct")
print("✓ tile < full matrix")
import numpy as np
_rng = np.random.default_rng(91)
for _t, _d, _dv in [(1, 3, 2), (7, 4, 3), (13, 5, 2)]:
    _q, _k, _v = _rng.normal(size=(_t, _d)), _rng.normal(size=(_t, _d)), _rng.normal(size=(_t, _dv))
    _s = _q @ _k.T / np.sqrt(_d)
    _s[np.triu_indices(_t, 1)] = -np.inf
    _p = np.exp(_s - _s.max(axis=1, keepdims=True))
    _expected = (_p / _p.sum(axis=1, keepdims=True)) @ _v
    for _b in [1, 2, 4, 20]:
        np.testing.assert_allclose(tiled_attention(_q, _k, _v, _b), _expected, atol=1e-10)
    _changed = _v.copy()
    _changed[-1] += 10000
    np.testing.assert_allclose(tiled_attention(_q, _k, _changed)[:-1], _expected[:-1], atol=1e-10)
# Large logits require running maxima, not raw exponentials.
_q = np.array([[1000.], [1001.], [1002.]])
_k = np.array([[1000.], [1002.], [1001.]])
np.testing.assert_allclose(tiled_attention(_q, _k, np.array([[1.], [2.], [3.]]), 1), [[1.], [2.], [2.]])
try:
    tiled_attention(_q, _k, _k, 0)
except ValueError:
    pass
else:
    raise AssertionError('block size 0 must raise ValueError')
print('Exact agreement, causal isolation, uneven tiles, and stable large logits passed.')
print("\n+200 XP — IO-Aware Attention complete.")


## Worked reference solution

Compare the state transitions and invariants with your implementation. Passing output tests alone does not establish memory efficiency or reproduce the paper's performance results.


In [ ]:
import numpy as np

def score_matrix_bytes(batch, heads, tokens, bytes_per_value):
    return batch * heads * tokens * tokens * bytes_per_value

def tile_bytes(query_rows, key_rows, head_size, bytes_per_value):
    return (query_rows * head_size + 2 * key_rows * head_size + query_rows * key_rows) * bytes_per_value

def tiled_attention(q, k, v, block_size=2):
    if block_size <= 0:
        raise ValueError('block_size must be positive')
    q, k, v = [np.asarray(x, dtype=np.float64) for x in (q, k, v)]
    t, d = q.shape
    output = np.empty((t, v.shape[1]))
    for start in range(0, t, block_size):
        stop = min(start + block_size, t)
        rows = np.arange(start, stop)
        m = np.full(stop - start, -np.inf)
        l = np.zeros(stop - start)
        a = np.zeros((stop - start, v.shape[1]))
        for col in range(0, stop, block_size):
            end = min(col + block_size, t)
            scores = q[start:stop] @ k[col:end].T / np.sqrt(d)
            scores = np.where(rows[:, None] >= np.arange(col, end), scores, -np.inf)
            # Old and new contributions must share the same exponential scale.
            new_m = np.maximum(m, scores.max(axis=1))
            correction = np.exp(m - new_m)
            p = np.exp(scores - new_m[:, None])
            l = correction * l + p.sum(axis=1)
            a = correction[:, None] * a + p @ v[col:end]
            m = new_m
        output[start:stop] = a / l[:, None]
    return output


In [ ]:
naive=score_matrix_bytes(1,1,4096,2)
assert naive==32*1024*1024,f"4096^2*2=32MiB, got {naive}"
tiled=tile_bytes(64,64,128,2)
assert tiled<naive,"one tile < full matrix"
assert tiled==64*128*2+64*128*2+64*128*2+64*64*2,f"tile bytes: {tiled}"
assert score_matrix_bytes(1,1,8192,2)==4*naive,"doubling tokens->4x memory"
print("✓ naive score matrix correct")
print("✓ tile < full matrix")
import numpy as np
_rng = np.random.default_rng(91)
for _t, _d, _dv in [(1, 3, 2), (7, 4, 3), (13, 5, 2)]:
    _q, _k, _v = _rng.normal(size=(_t, _d)), _rng.normal(size=(_t, _d)), _rng.normal(size=(_t, _dv))
    _s = _q @ _k.T / np.sqrt(_d)
    _s[np.triu_indices(_t, 1)] = -np.inf
    _p = np.exp(_s - _s.max(axis=1, keepdims=True))
    _expected = (_p / _p.sum(axis=1, keepdims=True)) @ _v
    for _b in [1, 2, 4, 20]:
        np.testing.assert_allclose(tiled_attention(_q, _k, _v, _b), _expected, atol=1e-10)
    _changed = _v.copy()
    _changed[-1] += 10000
    np.testing.assert_allclose(tiled_attention(_q, _k, _changed)[:-1], _expected[:-1], atol=1e-10)
# Large logits require running maxima, not raw exponentials.
_q = np.array([[1000.], [1001.], [1002.]])
_k = np.array([[1000.], [1002.], [1001.]])
np.testing.assert_allclose(tiled_attention(_q, _k, np.array([[1.], [2.], [3.]]), 1), [[1.], [2.], [2.]])
try:
    tiled_attention(_q, _k, _k, 0)
except ValueError:
    pass
else:
    raise AssertionError('block size 0 must raise ValueError')
print('Exact agreement, causal isolation, uneven tiles, and stable large logits passed.')
print("\n+200 XP — IO-Aware Attention complete.")


## Experiment and interpretation

Predict the result before running. Change one variable, repeat, and record what changed. Separate measured outcomes from assumptions.


In [ ]:
"""PyTorch tiled forward/backward agreement and optional explicit CUDA backend.

CPU: uv run --with torch python experiments/flash_torch.py
On CUDA, report median runtime and peak allocated bytes, not a universal speedup.
"""
import statistics
import time
import torch
import torch.nn.functional as F

torch.set_num_threads(1)

def torch_tiled(q, k, v, block=3):
    rows = []
    t, d = q.shape
    for start in range(0, t, block):
        end = min(start + block, t)
        m = torch.full((end-start,), -torch.inf, dtype=q.dtype, device=q.device)
        l = torch.zeros_like(m)
        a = torch.zeros(end-start, v.shape[-1], dtype=q.dtype, device=q.device)
        for col in range(0, end, block):
            stop = min(col+block, t)
            s = q[start:end] @ k[col:stop].T / d**.5
            allowed = torch.arange(start,end,device=q.device)[:,None] >= torch.arange(col,stop,device=q.device)
            s = s.masked_fill(~allowed, -torch.inf)
            new_m = torch.maximum(m, s.max(-1).values)
            correction = (m-new_m).exp()
            p = (s-new_m[:,None]).exp()
            a = correction[:,None]*a + p @ v[col:stop]
            l = correction*l + p.sum(-1)
            m = new_m
        rows.append(a/l[:,None])
    return torch.cat(rows)

def verify():
    torch.manual_seed(19)
    q,k,v = [torch.randn(7,4,dtype=torch.float64,requires_grad=True) for _ in range(3)]
    actual = torch_tiled(q,k,v)
    expected = F.scaled_dot_product_attention(q,k,v,is_causal=True)
    torch.testing.assert_close(actual,expected,atol=1e-10,rtol=1e-10)
    actual_grad = torch.autograd.grad(actual.square().sum(),(q,k,v),retain_graph=True)
    expected_grad = torch.autograd.grad(expected.square().sum(),(q,k,v))
    for a,b in zip(actual_grad,expected_grad):
        torch.testing.assert_close(a,b,atol=1e-9,rtol=1e-9)
    print('CPU float64 tiled output and Q/K/V gradients match SDPA.')

def cuda_benchmark():
    if not torch.cuda.is_available():
        print('CUDA unavailable: GPU backend, timing and peak-memory measurements skipped.')
        return
    from torch.nn.attention import SDPBackend, sdpa_kernel
    q,k,v = [torch.randn(1,4,512,64,device='cuda',dtype=torch.float16) for _ in range(3)]
    def explicit():
        s = q @ k.transpose(-1,-2) / 8
        return s.masked_fill(~torch.ones(512,512,device='cuda',dtype=torch.bool).tril(),-torch.inf).softmax(-1) @ v
    def flash():
        with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
            return F.scaled_dot_product_attention(q,k,v,is_causal=True)
    try:
        torch.testing.assert_close(flash(),explicit(),atol=.003,rtol=.003)
    except RuntimeError as error:
        print(f'Flash backend unavailable for this configuration: {error}')
        return
    print({'device':torch.cuda.get_device_name(), 'torch':torch.__version__, 'shape':list(q.shape),'dtype':str(q.dtype),'mode':'forward, no gradients'})
    for name, fn in [('explicit',explicit),('FLASH_ATTENTION backend',flash)]:
        for _ in range(5): fn()
        torch.cuda.synchronize()
        baseline = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
        samples=[]
        for _ in range(20):
            start=time.perf_counter(); fn(); torch.cuda.synchronize()
            samples.append((time.perf_counter()-start)*1000)
        print({'backend':name,'median_ms':statistics.median(samples),'p95_ms':sorted(samples)[18],
               'peak_extra_allocated_bytes':torch.cuda.max_memory_allocated()-baseline})

if __name__ == '__main__':
    verify()
    cuda_benchmark()
